# Pinion Guide Station — faithful 3D Gaussian Splat (free Colab, two-phase)

Free Colab's **GPU** quota is small (often ~2–3h), but COLMAP's slow step (the
SfM mapper) is **CPU-only**. So we split the work and never waste GPU time:

- **Phase 1 — CPU runtime:** COLMAP pose estimation → saved to Google Drive.
- **Phase 2 — GPU runtime:** load from Drive → train `splatfacto-big` → `.ply`.

Saving to Drive means a dropped session never costs you the SfM again, and
switching runtime types (which wipes `/content`) doesn't lose progress.

### How to run
1. **Runtime ▸ Change runtime type ▸ CPU.** Run **Cell 1** (mount Drive), then
   **Phase 1** = Cells 2–4. COLMAP finishes and saves `recon.zip` to Drive.
2. **Runtime ▸ Change runtime type ▸ GPU**, reconnect. Re-run **Cell 1** (mount
   Drive), then **Phase 2** = Cells 5–7. You get `splat.ply`.

**Input:** `pinion_panos.zip` — 192 rectilinear views sampled from 8 static
360° panoramas (`tools/panos_to_perspective.py --panos ...`, 8 yaws × 3
pitches each). Panos beat the walkthrough video here: the video had **moving
people/parts** that violate COLMAP's static-scene assumption and corrupt the
reconstruction. The panos are clean stills, so SfM is far more reliable.

> ⚠️ The 8 panos must be shot from **different standing positions** around the
> station. 3DGS needs parallax to triangulate depth — panos from a single
> tripod spot (pure rotation) cannot reconstruct geometry.

In [ ]:
# 1) Mount Google Drive. RUN THIS IN BOTH PHASES — re-run after switching the
#    runtime type (a fresh runtime unmounts Drive and wipes /content).
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/pinion_recon'
os.makedirs(DRIVE, exist_ok=True)
print('Drive workspace:', DRIVE)

In [ ]:
# ===== PHASE 1 (run on a CPU runtime) =====
# 2) System deps for COLMAP. CPU SIFT needs no GPU and no xvfb.
!apt-get -qq update
!apt-get -qq install -y colmap ffmpeg
!colmap -h | head -1

In [ ]:
# 3) Get the images. Uses pinion_panos.zip cached on Drive if present;
#    otherwise prompts an upload and caches it to Drive for next time.
#    (New filename so it never reuses the old video dataset by mistake.)
import os, zipfile, glob, shutil
shutil.rmtree('/content/recon/raw', ignore_errors=True)
shutil.rmtree('/content/recon/images', ignore_errors=True)
os.makedirs('/content/recon/images', exist_ok=True)
zip_drive = f'{DRIVE}/pinion_panos.zip'
if os.path.exists(zip_drive):
    zpath = zip_drive
    print('using cached pinion_panos.zip from Drive')
else:
    from google.colab import files
    up = files.upload()           # pick pinion_panos.zip
    zpath = next(iter(up))
    shutil.copy(zpath, zip_drive)
    print('cached pinion_panos.zip to Drive')
with zipfile.ZipFile(zpath) as z:
    z.extractall('/content/recon/raw')
for p in glob.glob('/content/recon/raw/**/*', recursive=True):
    if p.lower().endswith(('.jpg', '.jpeg', '.png')):
        shutil.copy(p, '/content/recon/images')
print('images:', len(os.listdir('/content/recon/images')))

In [ ]:
# 4) COLMAP directly on CPU (live progress), then save the result to Drive.
#    CPU SIFT is reliable on headless Colab. When it finishes, the
#    `model_analyzer` summary shows registered images + points.
import os, shutil
ROOT, IMG = '/content/recon', '/content/recon/images'
DB, SPARSE = f'{ROOT}/database.db', f'{ROOT}/colmap/sparse'
shutil.rmtree(f'{ROOT}/colmap', ignore_errors=True)   # drop any stale (video) model
os.makedirs(SPARSE, exist_ok=True)
!rm -f "$DB"

print('== 1/3 feature extraction (CPU) ==')
# PINHOLE: the pano crops are undistorted shared-intrinsic pinhole views, so no
# lens-distortion params to solve — a tighter, more stable calibration.
# NOTE: keep use_gpu 0 everywhere. Colab's COLMAP is a CPU-only build AND its
# SiftGPU needs an OpenGL display (absent on headless Colab), so use_gpu 1 hangs.
# max_num_features caps SIFT keypoints/image -> faster matching + mapper, plenty
# for these high-res pano crops.
!colmap feature_extractor \
    --database_path "$DB" --image_path "$IMG" \
    --ImageReader.single_camera 1 \
    --ImageReader.camera_model PINHOLE \
    --SiftExtraction.use_gpu 0 \
    --SiftExtraction.max_num_features 6000

print('== 2/3 exhaustive matching (CPU) ==')
# 192 images -> exhaustive (all pairs) is tractable on CPU and finds more
# matches than vocab-tree, which matters with only 8 capture positions.
!colmap exhaustive_matcher \
    --database_path "$DB" \
    --SiftMatching.use_gpu 0

print('== 3/3 mapper / SfM ==')
!colmap mapper \
    --database_path "$DB" --image_path "$IMG" --output_path "$SPARSE"

print('== result (registered images + points) ==')
!ls -la "$SPARSE/0" && colmap model_analyzer --path "$SPARSE/0"

# Save images + colmap model to Drive (excludes the big database.db) so the GPU
# phase resumes exactly here, even on a fresh runtime.
print('== saving recon.zip to Drive ==')
!cd /content/recon && zip -qr "{DRIVE}/recon.zip" images colmap && echo "saved {DRIVE}/recon.zip"

In [ ]:
# ===== PHASE 2 (switch Runtime to GPU now, then re-run Cell 1 to remount Drive) =====
# 5) Install nerfstudio + gsplat (the 3DGS trainer + rasterizer). ~5-10 min.
#    The pip dependency-conflict warnings are harmless; wait for "nerfstudio installed".
!pip -q install nerfstudio
!pip -q install gsplat
print("nerfstudio installed")

In [ ]:
# 6) Restore COLMAP output from Drive (fresh GPU runtime), then train.
#    `splatfacto-big` = higher quality (more Gaussians) for the final splat.
#    Trainer args go BEFORE the `colmap` dataparser subcommand.
import os, zipfile
if not os.path.exists('/content/recon/colmap/sparse/0'):
    with zipfile.ZipFile(f'{DRIVE}/recon.zip') as z:
        z.extractall('/content/recon')
    print('restored recon from Drive')
print('images:', len(os.listdir('/content/recon/images')))

!ns-train splatfacto-big \
    --max-num-iterations 30000 \
    --viewer.quit-on-train-completion True \
    --output-dir /content/recon/outputs \
    colmap --data /content/recon

In [ ]:
# 7) Export the trained splat to .ply by reading the checkpoint directly.
#    `ns-export` tends to hang/crash on Colab, so we load the .ckpt and write
#    the .ply ourselves — this is the path that worked reliably last time.
#
#    If torch.load raises a numpy error ("Weights only load failed" /
#    "scalar() argument ... numpy.dtypes.Float64DType"), the ckpt was saved with
#    numpy 1.x but the runtime has numpy 2.x. Fix once:
#      !pip install -q "numpy<2"   ->  Runtime ▸ Restart session  ->  re-run
#      Cell 1 (remount Drive) and this cell.
import torch, numpy as np, glob, os, shutil

ckpt = sorted(glob.glob('/content/recon/outputs/**/*.ckpt', recursive=True))[-1]
print('loading', ckpt, flush=True)
sd = torch.load(ckpt, map_location='cpu', weights_only=False)['pipeline']

def g(name):
    for k in sd:
        if k.endswith('gauss_params.' + name):
            return sd[k].float().cpu().numpy()
    raise KeyError(name)

means  = g('means')                      # (N,3)
scales = g('scales')                     # (N,3) log
quats  = g('quats')                      # (N,4) wxyz
opac   = g('opacities').reshape(-1, 1)   # (N,1) logit
fdc    = g('features_dc')                # (N,3)
frest  = g('features_rest')              # (N,K,3)
N, K = means.shape[0], frest.shape[1]
print('gaussians:', N, ' sh_rest_coeffs:', K, flush=True)

frest_cm = np.transpose(frest, (0, 2, 1)).reshape(N, 3 * K)  # channel-major

cols, names = [], []
for i, n in enumerate(['x', 'y', 'z']): cols.append(means[:, i]); names.append(n)
for n in ['nx', 'ny', 'nz']: cols.append(np.zeros(N, np.float32)); names.append(n)
for i in range(3): cols.append(fdc[:, i]); names.append(f'f_dc_{i}')
for i in range(3 * K): cols.append(frest_cm[:, i]); names.append(f'f_rest_{i}')
cols.append(opac[:, 0]); names.append('opacity')
for i in range(3): cols.append(scales[:, i]); names.append(f'scale_{i}')
for i in range(4): cols.append(quats[:, i]); names.append(f'rot_{i}')

data = np.stack(cols, axis=1).astype('<f4')
out = '/content/recon/export/pinion_splat.ply'
os.makedirs(os.path.dirname(out), exist_ok=True)
with open(out, 'wb') as f:
    hdr = 'ply\nformat binary_little_endian 1.0\n' + f'element vertex {N}\n'
    hdr += ''.join(f'property float {nm}\n' for nm in names) + 'end_header\n'
    f.write(hdr.encode('ascii')); f.write(data.tobytes())
print('WROTE', out, round(os.path.getsize(out) / 1e6, 1), 'MB', flush=True)

# save to Drive + download
os.makedirs('/content/drive/MyDrive/pinion_recon', exist_ok=True)
shutil.copy(out, '/content/drive/MyDrive/pinion_recon/pinion_splat.ply')
print('saved to Drive: /content/drive/MyDrive/pinion_recon/pinion_splat.ply', flush=True)
from google.colab import files; files.download(out)

## Done — hand me the `.ply`

Send back the downloaded `splat.ply` (also saved on Drive as
`pinion_recon/pinion_splat.ply`). The Spark viewer loads `.ply` directly, so I
drop it into the synthetic-pov top-right quadrant (no World Labs).

**Optional — smaller file:** convert `.ply` → `.spz` (often 5–10× smaller) via
<https://spz-to-ply.netlify.app>. Either format works in the viewer.

**Reading the COLMAP summary (Cell 4):** check `Registered images` out of 192.
- 150+ → strong; train as-is.
- ~90–150 → usable; the panos likely had decent overlap.
- < ~90 or no `sparse/0` → too few panos / not enough overlap or parallax
  between capture spots; re-shoot with more positions and ~50% overlap.

**Why two phases:** COLMAP's mapper is CPU-only, so running it on a CPU runtime
keeps it off the scarce GPU quota; only `splatfacto-big` training needs the GPU.